# 03 — Kerr geodesics and Carter's constant

**Spacetime Lab — Phase 3 concept + implementation notebook**

This notebook is the companion to `spacetime_lab.metrics.Kerr` and `spacetime_lab.geodesics.GeodesicIntegrator`. It walks through the physics of rotating black holes — the line element, horizons, ergoregion, ISCO/photon-sphere splittings — and then demonstrates the central new piece of physics: **Carter's constant is conserved along Kerr geodesics**, despite not coming from any continuous symmetry.

**What you will learn**

1. The Kerr line element in Boyer–Lindquist coordinates and its $\Sigma$, $\Delta$ abbreviations
2. Outer event horizon $r_+$, inner Cauchy horizon $r_-$, and the ergoregion between $r_+$ and the static limit
3. Why the ISCO and photon sphere split into prograde/retrograde branches when $a \neq 0$
4. Horizon thermodynamics for Kerr: $\Omega_H$, $\kappa$, $T_H$, $S_{BH}$, and the *third law* (extremal Kerr has $T = 0$)
5. Hamilton's equations for geodesic motion as the action of the inverse metric on covariant momenta
6. The implicit-midpoint symplectic integrator and *why* it preserves $E$ and $L_z$ exactly
7. **Carter's constant** as the irreducible Killing-tensor invariant — and verifying it numerically along an integrated orbit

**Conventions**: signature $(-,+,+,+)$, geometric units $G = c = 1$, $a = J/M$ with $a \in [0, M]$.

**References**

- Wald, *General Relativity*, ch. 12
- Bardeen, Press & Teukolsky, *ApJ* **178** 347 (1972)
- Carter, *Phys. Rev.* **174** 1559 (1968)
- [Wikipedia: Kerr metric](https://en.wikipedia.org/wiki/Kerr_metric)

In [ ]:
import math
import numpy as np
import matplotlib.pyplot as plt

from spacetime_lab.metrics import Kerr
from spacetime_lab.geodesics import GeodesicIntegrator, GeodesicState

plt.rcParams['figure.figsize'] = (7, 4.5)

## 1. Build a Kerr black hole

We use $M = 1$ and a moderate spin $a = 0.5$. Geometric units mean every length is in units of $M$, so 'the ISCO is at $r = 4.23\,M$' becomes literally 'the ISCO is at $r = 4.23$'.

In [ ]:
bh = Kerr(mass=1.0, spin=0.5)
print(bh)
print()
print(f'outer horizon r_+      = {bh.outer_horizon():.6f}')
print(f'inner horizon r_-      = {bh.inner_horizon():.6f}')
print(f'ergosphere at equator  = {bh.ergosphere(math.pi / 2):.6f}')
print(f'ergosphere at theta=pi/4 = {bh.ergosphere(math.pi / 4):.6f}')
print()
print(f'ISCO prograde          = {bh.isco(prograde=True):.6f}')
print(f'ISCO retrograde        = {bh.isco(prograde=False):.6f}')
print(f'photon sphere prograde = {bh.photon_sphere_equatorial(True):.6f}')
print(f'photon sphere retrograde= {bh.photon_sphere_equatorial(False):.6f}')
print()
print(f'Omega_H                = {bh.angular_velocity_horizon():.6f}')
print(f'horizon area           = {bh.horizon_area():.6f}')
print(f'surface gravity kappa  = {bh.surface_gravity():.6f}')
print(f'Hawking temperature    = {bh.hawking_temperature():.6f}')
print(f'Bekenstein-Hawking S   = {bh.bekenstein_hawking_entropy():.6f}')

## 2. The horizon, the ergosphere, and the static limit

Three surfaces matter for any rotating black hole:

- **Outer horizon** $r_+ = M + \sqrt{M^2 - a^2}$ — the actual one-way membrane.
- **Inner Cauchy horizon** $r_- = M - \sqrt{M^2 - a^2}$ — the surface beyond which the spacetime stops being predictable from initial data on a Cauchy slice. New compared to Schwarzschild.
- **Static limit** $r_E(\theta) = M + \sqrt{M^2 - a^2 \cos^2\theta}$ — the surface inside which $\partial_t$ becomes spacelike, so no observer can remain at rest. Coincides with $r_+$ at the poles and bulges out to $2M$ at the equator. Between $r_+$ and $r_E$ is the **ergoregion**, where the Penrose process can extract rotational energy from the hole.

In [ ]:
# Plot the three surfaces in the (x, z) half-plane (axisymmetric, so we
# parametrise by colatitude theta and use cylindrical x = r sin(theta),
# z = r cos(theta)).
thetas = np.linspace(0.001, math.pi - 0.001, 200)

r_plus = bh.outer_horizon() * np.ones_like(thetas)
r_minus = bh.inner_horizon() * np.ones_like(thetas)
r_erg = np.array([bh.ergosphere(t) for t in thetas])

def to_xz(rs, ths):
    return rs * np.sin(ths), rs * np.cos(ths)

fig, ax = plt.subplots(figsize=(6, 6))
x, z = to_xz(r_plus, thetas)
ax.fill(np.concatenate([x, -x[::-1]]), np.concatenate([z, z[::-1]]), color='#222222', alpha=0.4, label='outer horizon r_+')
x, z = to_xz(r_minus, thetas)
ax.fill(np.concatenate([x, -x[::-1]]), np.concatenate([z, z[::-1]]), color='#a40000', alpha=0.4, label='inner horizon r_-')
x, z = to_xz(r_erg, thetas)
ax.plot(x, z, color='#0050a0', lw=1.5, label='static limit r_E')
ax.plot(-x, z, color='#0050a0', lw=1.5)
ax.axhline(0, color='k', lw=0.4)
ax.axvline(0, color='k', lw=0.4)
ax.set_aspect('equal')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_xlabel('x = r sin(theta)')
ax.set_ylabel('z = r cos(theta)')
ax.set_title(f'Kerr a={bh.spin}: horizons and static limit')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

Notice that the ergoregion (the lens-shaped sliver between the dark grey outer horizon and the blue static limit) **only exists off-axis**. At the poles the static limit touches $r_+$, so there is no ergoregion there; at the equator it extends out to $2M$.

## 3. ISCO and photon sphere — prograde versus retrograde

Schwarzschild's spherical symmetry forces the ISCO to be a single sphere at $r = 6M$. Kerr breaks this. Frame-dragging adds a centripetal effect for prograde orbits and a centrifugal one for retrograde, so:

- The **prograde** ISCO is *smaller* than the Schwarzschild value, monotonically decreasing as $a \to M$, and at extremal Kerr it touches the horizon at $r = M$.
- The **retrograde** ISCO is *larger*, monotonically increasing, and reaches $r = 9M$ at extremal Kerr.
- The same split happens for the equatorial photon sphere: $r = 3M$ at $a = 0$, splitting to $M$ (prograde) and $4M$ (retrograde) at extremal Kerr.

In [ ]:
spins = np.linspace(0.0, 0.999, 100)
isco_pro = [Kerr(mass=1.0, spin=a).isco(prograde=True) for a in spins]
isco_retro = [Kerr(mass=1.0, spin=a).isco(prograde=False) for a in spins]
ph_pro = [Kerr(mass=1.0, spin=a).photon_sphere_equatorial(True) for a in spins]
ph_retro = [Kerr(mass=1.0, spin=a).photon_sphere_equatorial(False) for a in spins]
rp = [Kerr(mass=1.0, spin=a).outer_horizon() for a in spins]

fig, ax = plt.subplots()
ax.plot(spins, isco_pro, color='#1a9a4a', lw=1.6, label='ISCO prograde')
ax.plot(spins, isco_retro, color='#1a9a4a', lw=1.6, ls='--', label='ISCO retrograde')
ax.plot(spins, ph_pro, color='#c64a0b', lw=1.6, label='photon sphere prograde')
ax.plot(spins, ph_retro, color='#c64a0b', lw=1.6, ls='--', label='photon sphere retrograde')
ax.plot(spins, rp, color='#000000', lw=1.2, label='outer horizon r_+')
ax.set_xlabel('spin a / M')
ax.set_ylabel('radius / M')
ax.set_title('Equatorial circular orbits in Kerr')
ax.legend(loc='upper right', fontsize=9)
plt.tight_layout()
plt.show()

All five curves meet at the Schwarzschild values ($6, 6, 3, 3, 2$) when $a = 0$, and at extremal Kerr the prograde ISCO and prograde photon sphere both touch the horizon at $r = M$.

## 4. The strong test of the line element: $R_{\mu\nu} = 0$

Before we trust this metric for integrating geodesics, we need to know it actually solves the *vacuum* Einstein equations. If any one of the four non-trivial $g_{\mu\nu}$ components were miswritten, the Ricci tensor would not vanish.

Symbolically computing $R_{\mu\nu}$ for Kerr is *pathologically slow* in sympy (each `simplify` call can take minutes). The trick is to lambdify everything to numerical functions and evaluate at sample points instead. The Kerr class exposes `verify_vacuum_numerical()` which does exactly this:

In [ ]:
import time
for a in [0.0, 0.3, 0.5, 0.9]:
    k = Kerr(mass=1.0, spin=a)
    t0 = time.time()
    max_R = k.verify_vacuum_numerical()
    elapsed = time.time() - t0
    print(f'  a = {a:.2f}   max |R_munu| = {max_R:.3e}   ({elapsed:.1f}s)')

Pure floating-point noise — the line element is correct and Kerr is genuinely vacuum.

## 5. Hamilton's equations for geodesics

The geodesic equation can be written as Hamilton's equations for the **Hamiltonian**

$$H(x, p) = \tfrac{1}{2}\, g^{\mu\nu}(x)\, p_\mu p_\nu,$$

with covariant momentum $p_\mu = g_{\mu\nu} \dot x^\nu$. Equations of motion:

$$\frac{dx^\mu}{d\lambda} = g^{\mu\nu}(x)\, p_\nu, \qquad \frac{dp_\mu}{d\lambda} = -\tfrac{1}{2}\, \partial_\mu g^{\alpha\beta}(x)\, p_\alpha p_\beta.$$

Two important things:

1. **The Hamiltonian itself is conserved** along any flow line, which gives the mass-shell constraint $-2H = \mu^2$ ($\mu = 1$ for a normalised massive particle, $\mu = 0$ for a photon).
2. **Cyclic coordinates have exactly conserved conjugate momenta**. Since Boyer–Lindquist Kerr is independent of $t$ and $\varphi$ ($\partial_t H = \partial_\varphi H = 0$), the components $p_t$ and $p_\varphi$ are *exactly* conserved by the equations of motion. We identify:
$$E \equiv -p_t \quad (\text{energy at infinity}), \qquad L_z \equiv +p_\varphi \quad (\text{axial angular momentum}).$$

This gives 3 conserved quantities: $E$, $L_z$, and the Hamiltonian. We need one more for full integrability (Liouville's theorem requires 4 in involution on an 8-dimensional phase space). That fourth one is **Carter's constant**.

## 6. The implicit-midpoint symplectic integrator

The kinetic term $\tfrac{1}{2} g^{\mu\nu}(x) p_\mu p_\nu$ depends on the position $x$, so the Hamiltonian is **non-separable**. Standard leapfrog (which assumes $H = T(p) + V(x)$) does not apply. We use **implicit midpoint** instead:

$$x_{n+1} = x_n + h \cdot \dot x\!\left(\tfrac{x_n + x_{n+1}}{2}, \tfrac{p_n + p_{n+1}}{2}\right),$$
$$p_{n+1} = p_n + h \cdot \dot p\!\left(\tfrac{x_n + x_{n+1}}{2}, \tfrac{p_n + p_{n+1}}{2}\right).$$

This is the one-stage Gauss–Legendre method: symplectic, time-reversible, 2nd-order accurate, and works for non-separable Hamiltonians. The system is solved at every step by `scipy.optimize.fsolve`.

**Crucial property**: cyclic coordinates have their conjugate momenta preserved to *machine precision*, because $\dot p_\alpha = 0$ identically when $\partial_\alpha g^{\mu\nu} = 0$, and the implicit midpoint method respects this exactly. So **$E$ and $L_z$ will be conserved to ~$10^{-15}$**, while $H$ and Carter's $\mathcal{Q}$ will only be conserved to $O(h^2)$ per step. The drift of the latter two is the diagnostic of how faithfully we are following geodesics.

In [ ]:
# Pick a generic timelike-ish initial state outside the horizon, with
# both p_theta and p_phi non-zero so the orbit is genuinely off-equator
# and Carter's Q is non-trivial.
state0 = GeodesicState(
    x=np.array([0.0, 10.0, 1.4, 0.0]),     # (t, r, theta, phi)
    p=np.array([-0.97, 0.0, 1.5, 3.0]),    # (p_t, p_r, p_theta, p_phi)
)

integrator = GeodesicIntegrator(bh)
init = bh.constants_of_motion(state0)
print('Initial constants of motion:')
for k, v in init.items():
    print(f'  {k:12s} = {v: .8f}')

In [ ]:
# Integrate 400 implicit-midpoint steps with h = 0.5 (lambda = 200)
import time
t0 = time.time()
states = integrator.integrate(state0, h=0.5, n_steps=400)
print(f'integrated 400 steps in {time.time() - t0:.1f}s')

lambdas = np.arange(len(states)) * 0.5
Es   = np.array([-s.p[0] for s in states])
Lzs  = np.array([s.p[3] for s in states])
Hs   = np.array([integrator.hamiltonian(s) for s in states])
Qs   = np.array([bh.carter_constant_from_state(s) for s in states])

In [ ]:
# Plot relative deviation of each conserved quantity from its initial value.
fig, ax = plt.subplots(figsize=(8, 5))

def rel_dev(arr):
    return np.abs(arr - arr[0]) / np.maximum(np.abs(arr[0]), 1e-300)

ax.plot(lambdas, np.maximum(rel_dev(Es), 1e-17), label='E (= -p_t)', color='#0050a0')
ax.plot(lambdas, np.maximum(rel_dev(Lzs), 1e-17), label='L_z (= p_phi)', color='#1a9a4a')
ax.plot(lambdas, np.maximum(rel_dev(np.abs(Hs)), 1e-17), label='|H| = mu^2 / 2', color='#c64a0b')
ax.plot(lambdas, np.maximum(rel_dev(Qs), 1e-17), label='Q (Carter)', color='#a40000')
ax.set_yscale('log')
ax.set_xlabel('affine parameter lambda')
ax.set_ylabel('relative deviation from initial value')
ax.set_title(f'Conservation along Kerr (a={bh.spin}) geodesic')
ax.set_ylim(1e-17, 1e-2)
ax.axhline(1e-15, color='gray', ls=':', lw=0.8)
ax.text(lambdas[-1], 1.5e-15, 'machine precision', color='gray', fontsize=8, ha='right')
ax.legend(loc='lower right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

**Read the plot like this:**

- The blue and green lines (the Killing vector quantities $E$ and $L_z$) sit at machine precision $\sim 10^{-15}$ for the entire integration. The integrator preserves them exactly because $t$ and $\varphi$ are cyclic.
- The orange ($H$) and red (Carter's $\mathcal{Q}$) lines drift at the same order of magnitude — both are $O(h^2)$ per step. They oscillate rather than monotonically diverging, which is the *symplectic* signature: total energy isn't exactly conserved but stays bounded forever.
- The fact that **Carter's $\mathcal{Q}$ is conserved to the same precision as $H$**, despite there being no Killing vector for it, is the experimental confirmation that the irreducible Killing tensor is real and our polynomial-form Carter's constant is correct.

## 7. Convergence test: 2nd-order in $h$

If the integrator is genuinely 2nd-order, halving the step size should reduce the drift in $\mathcal{Q}$ by a factor of 4.

In [ ]:
total_lambda = 50.0
step_sizes = [1.0, 0.5, 0.25, 0.125]
drifts_Q = []
drifts_H = []
for h in step_sizes:
    n = int(round(total_lambda / h))
    sts = integrator.integrate(state0, h=h, n_steps=n)
    Qs = np.array([bh.carter_constant_from_state(s) for s in sts])
    Hs = np.array([integrator.hamiltonian(s) for s in sts])
    drifts_Q.append(np.max(np.abs(Qs - Qs[0])) / abs(Qs[0]))
    drifts_H.append(np.max(np.abs(Hs - Hs[0])) / abs(Hs[0]))

print(f'{"h":>8s} {"steps":>6s} {"drift Q":>12s} {"drift H":>12s} {"Q ratio":>10s}')
for i, h in enumerate(step_sizes):
    n = int(round(total_lambda / h))
    if i == 0:
        ratio_str = '  -'
    else:
        ratio_str = f'{drifts_Q[i-1] / drifts_Q[i]:7.2f}'
    print(f'{h:8.4f} {n:6d} {drifts_Q[i]:12.3e} {drifts_H[i]:12.3e} {ratio_str:>10s}')

Each row shows the drift falling by a factor of $\sim 4$ when $h$ halves — exactly $h^2$. That's the smoking-gun signature of a 2nd-order method.

## 8. The orbit in (r, theta) plane

Just so we know what the geodesic actually looks like: project onto the $(r \sin\theta,\, r \cos\theta)$ plane, ignoring $\varphi$. A non-zero $\mathcal{Q}$ produces an oscillation in $\theta$ that traces out a band, while $r$ oscillates between two turning points. The combination is the famous *spherical photon orbit / spherical timelike orbit* of Kerr.

In [ ]:
rs = np.array([s.x[1] for s in states])
ths = np.array([s.x[2] for s in states])
x_proj = rs * np.sin(ths)
z_proj = rs * np.cos(ths)

fig, ax = plt.subplots(figsize=(7, 7))
ax.plot(x_proj, z_proj, color='#0050a0', lw=0.6)
# Overlay the horizon and ergosphere
thetas = np.linspace(0.001, math.pi - 0.001, 200)
rp_circle = bh.outer_horizon() * np.ones_like(thetas)
rerg = np.array([bh.ergosphere(t) for t in thetas])
ax.fill(np.concatenate([rp_circle * np.sin(thetas), -(rp_circle * np.sin(thetas))[::-1]]),
        np.concatenate([rp_circle * np.cos(thetas), (rp_circle * np.cos(thetas))[::-1]]),
        color='#222222', alpha=0.5, label='r_+')
ax.plot(rerg * np.sin(thetas), rerg * np.cos(thetas), color='#0050a0', ls='--', lw=1.0, label='static limit')
ax.plot(-rerg * np.sin(thetas), rerg * np.cos(thetas), color='#0050a0', ls='--', lw=1.0)
ax.set_aspect('equal')
ax.set_xlim(-12, 12)
ax.set_ylim(-12, 12)
ax.set_xlabel('r sin(theta)')
ax.set_ylabel('r cos(theta)')
ax.set_title('Off-equator Kerr geodesic (projected to r-theta plane)')
ax.legend(loc='upper right', fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Phase 3 gate checks

Before we move on to Phase 4 the following must hold. The assertions below fail loudly if anything regresses.

In [ ]:
# (1) Schwarzschild limit of Kerr: every quantity collapses at a = 0
k0 = Kerr(mass=1.0, spin=0.0)
assert math.isclose(k0.outer_horizon(), 2.0)
assert math.isclose(k0.inner_horizon(), 0.0)
assert math.isclose(k0.isco(True), 6.0)
assert math.isclose(k0.isco(False), 6.0)
assert math.isclose(k0.photon_sphere_equatorial(True), 3.0)
assert math.isclose(k0.surface_gravity(), 0.25)

# (2) Extremal Kerr: third law and prograde-touching-horizon
ke = Kerr(mass=1.0, spin=1.0)
assert math.isclose(ke.outer_horizon(), 1.0)
assert math.isclose(ke.inner_horizon(), 1.0)
assert math.isclose(ke.isco(True), 1.0, abs_tol=1e-9)
assert math.isclose(ke.isco(False), 9.0, abs_tol=1e-9)
assert math.isclose(ke.surface_gravity(), 0.0, abs_tol=1e-12)
assert math.isclose(ke.angular_velocity_horizon(), 0.5)

# (3) Vacuum Einstein for the metric we use throughout the notebook
assert bh.verify_vacuum_numerical() < 1e-10

# (4) Killing-vector momenta exactly conserved by the integrator
assert np.max(np.abs(Es - Es[0])) < 1e-12
assert np.max(np.abs(Lzs - Lzs[0])) < 1e-12

# (5) Hamiltonian and Carter's Q drift only at O(h^2) order
assert np.max(np.abs(Hs - Hs[0])) / abs(Hs[0]) < 1e-3
assert np.max(np.abs(Qs - Qs[0])) / abs(Qs[0]) < 1e-3

# (6) Equatorial-orbit Carter formula gives Q = 0 identically
Q_eq = bh.carter_constant(E=0.95, L_z=3.0, p_theta=0.0, theta=math.pi / 2)
assert math.isclose(Q_eq, 0.0, abs_tol=1e-12)

# (7) 2nd-order convergence (Q drift falls by ~4x when h halves)
for i in range(1, len(step_sizes)):
    ratio = drifts_Q[i - 1] / drifts_Q[i]
    assert 2.0 < ratio < 8.0, f'h-convergence ratio {ratio} outside (2, 8)'

print('ALL PHASE 3 GATE CHECKS PASSED')

---

## What's next — Phase 4 teaser

Phase 4 builds the **horizon finders** module: numerical detection of apparent horizons and event horizons from a generic initial-data slice. This is where Phase 3's geodesic integrator earns its keep — the event horizon of a dynamic spacetime is *defined* as the past boundary of the union of future-directed null geodesics that escape to infinity, so finding it numerically is a global integration problem. We will reuse `GeodesicIntegrator` with photon initial conditions ($\mu^2 = 0$) and shoot rays inward from $\mathscr{I}^+$.

The other Phase 4 deliverables are: marginally outer-trapped surface (MOTS) finder, ISCO finder for arbitrary metrics (not just analytical Kerr), photon-region computation for non-equatorial Kerr orbits, and a notebook reproducing the Bardeen photon shadow that EHT measures.